<a href="https://colab.research.google.com/github/amanda-carvalhosc/otimizacao-logistica-rj/blob/main/analise_e_limpeza_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np

# 1. Importação da base de dados bruta
df = pd.read_excel('entregas_bruto.xlsx')

# 2. Padronização e higienização dos nomes dos colaboradores (Motoristas)
df['Motorista'] = df['Motorista'].astype(str).str.strip().str.title()

# 3. Tratamento de inconsistências de texto e encoding na coluna Regiao_Destino
df['Regiao_Destino'] = df['Regiao_Destino'].astype(str).str.strip()
df.loc[df['Regiao_Destino'].str.contains('Sao|São|Gon|þ', case=False, na=False), 'Regiao_Destino'] = 'São Gonçalo'
df.loc[df['Regiao_Destino'].str.contains('Nit', case=False, na=False), 'Regiao_Destino'] = 'Niterói'
df.loc[df['Regiao_Destino'].str.contains('Rio|Janeiro', case=False, na=False), 'Regiao_Destino'] = 'Rio de Janeiro'

# 4. Tratamento de falha de encoding na variável Status_Entrega
df['Status_Entrega'] = df['Status_Entrega'].astype(str).str.strip()
df.loc[df['Status_Entrega'].str.contains('tr|Æ|nsito', case=False, na=False), 'Status_Entrega'] = 'Em trânsito'

# 5. Conversão e garantia dos tipos de dados numéricos (Cast)
df['Distancia_KM'] = pd.to_numeric(df['Distancia_KM'], errors='coerce')
df['Peso_Carga_KG'] = pd.to_numeric(df['Peso_Carga_KG'], errors='coerce')
df['Capacidade_Veiculo_KG'] = pd.to_numeric(df['Capacidade_Veiculo_KG'], errors='coerce')

# 6. Inputação de valores ausentes (NaN) em Custo_Combustivel com base na média ponderada por KM
custo_por_km_medio = (df['Custo_Combustivel'] / df['Distancia_KM']).mean()
df['Custo_Combustivel'] = df['Custo_Combustivel'].fillna(df['Distancia_KM'] * custo_por_km_medio)
df['Custo_Combustivel'] = df['Custo_Combustivel'].round(2)

# 7. Tratamento final de registros ausentes no status da operação
df['Status_Entrega'] = df['Status_Entrega'].replace('nan', 'Não Informado')

print("--- BASE DE DADOS HIGIENIZADA E PADRONIZADA ---")
df


--- BASE DE DADOS HIGIENIZADA E PADRONIZADA ---


,ID_Entrega,Motorista,Regiao_Destino,Distancia_KM,Custo_Combustivel,Status_Entrega,Capacidade_Veiculo_KG,Peso_Carga_KG
0,ENT-001,Carlos Silva,São Gonçalo,25,120.5,Em trânsito,1000,850
1,ENT-002,Mariana Souza,Niterói,42,210.0,Não Informado,1500,1600
2,ENT-003,Carlos Silva,São Gonçalo,18,85.0,Em trânsito,1000,900
3,ENT-004,Roberto Lima,Rio de Janeiro,65,315.4,Em trânsito,2000,1100
4,ENT-005,Mariana Souza,Niterói,38,190.0,Cancelado,1500,400
5,ENT-006,Roberto Lima,Rio de Janeiro,70,320.0,Em trânsito,2000,2150
6,ENT-007,Carlos Silva,São Gonçalo,12,60.0,Em trânsito,1000,300


In [7]:
print("=== RELATÓRIO GERENCIAL DE LOGÍSTICA E OPERAÇÕES ===\n")

# 1. Cálculo do custo total acumulado da operação de transporte
custo_total = df['Custo_Combustivel'].sum()
print(f"1. Custo Total de Transporte (Combustível): R$ {custo_total:,.2f}\n")

# 2. Identificação de inconformidades operacionais (Sobrecarga de capacidade de carga)
print("2. Auditoria de Capacidade de Carga da Frota:")
motoristas_sobrecarregados = df[df['Peso_Carga_KG'] > df['Capacidade_Veiculo_KG']]

if len(motoristas_sobrecarregados) > 0:
    for index, linha in motoristas_sobrecarregados.iterrows():
        print(f"   [ALERTA OPERACIONAL] Motorista: {linha['Motorista']} | ID: {linha['ID_Entrega']} | Carga Real: {linha['Peso_Carga_KG']} KG | Capacidade Máxima: {linha['Capacidade_Veiculo_KG']} KG")
else:
    print("   -> Conformidade atingida: nenhuma operação acima da capacidade máxima permitida.")
print("\n")

# 3. Agrupamento e análise de custos logísticos por localidade de destino
print("3. Consolidação de Custos por Região Geográfica:")
custo_por_regiao = df.groupby('Regiao_Destino')['Custo_Combustivel'].sum().reset_index()

for index, linha in custo_por_regiao.iterrows():
    print(f"   -> {linha['Regiao_Destino']}: R$ {linha['Custo_Combustivel']:,.2f}")

regiao_mais_cara = custo_por_regiao.sort_values(by='Custo_Combustivel', ascending=False).iloc[0]
print(f"\n📢 IMPACTO FINANCEIRO: A localidade com maior centro de custo é {regiao_mais_cara['Regiao_Destino']} (R$ {regiao_mais_cara['Custo_Combustivel']:,.2f})")

# 4. Exportação da base de dados tratada para o diretório local
df.to_excel('entregas_limpas_operacao.xlsx', index=False)
print("\n[INFO] Arquivo 'entregas_limpas_operacao.xlsx' exportado com sucesso para conexão com o Power BI.")


=== RELATÓRIO GERENCIAL DE LOGÍSTICA E OPERAÇÕES ===

1. Custo Total de Transporte (Combustível): R$ 1,300.90

2. Auditoria de Capacidade de Carga da Frota:
   [ALERTA OPERACIONAL] Motorista: Mariana Souza | ID: ENT-002 | Carga Real: 1600 KG | Capacidade Máxima: 1500 KG
   [ALERTA OPERACIONAL] Motorista: Roberto Lima | ID: ENT-006 | Carga Real: 2150 KG | Capacidade Máxima: 2000 KG


3. Consolidação de Custos por Região Geográfica:
   -> Niterói: R$ 400.00
   -> Rio de Janeiro: R$ 635.40
   -> São Gonçalo: R$ 265.50

📢 IMPACTO FINANCEIRO: A localidade com maior centro de custo é Rio de Janeiro (R$ 635.40)

[INFO] Arquivo 'entregas_limpas_operacao.xlsx' exportado com sucesso para conexão com o Power BI.
